# WEEK 9｜Pandas 数据合并、分组与透视

source 介绍数组和表格拼接、按键合并、GroupBy 分组计算，以及用透视表汇总数据。

## 1. Concatenate：沿轴拼接

`pd.concat` 把多个 Series 或 DataFrame 沿行轴或列轴连接。按行连接是默认方向；按列连接可传 `axis=1`。`ignore_index=True` 可在结果中重建行序号；`keys` 可为拼接来源增加层级标签。

In [ ]:
import pandas as pd

left = pd.DataFrame({"name": ["Amy", "Bo"], "score": [80, 90]})
right = pd.DataFrame({"name": ["Cy"], "score": [85]})
rows = pd.concat([left, right], ignore_index=True)
print(rows)

extra = pd.Series([1, 2, 3], name="attempt")
combined = pd.concat([left, extra], axis=1)
print(combined)

## 2. Merge：按键匹配记录

`merge` 根据共同键或指定键匹配两张表的记录。inner 保留两侧都匹配的键；left 保留左表所有键；right 保留右表所有键；outer 保留两侧键的并集。可用 `on` 指定同名键，或用 `left_on` 与 `right_on` 指定不同列名。

In [ ]:
students1 = pd.DataFrame({
    "student": ["Amy", "Bo", "Cy"],
    "class": ["A", "A", "B"],
})
students2 = pd.DataFrame({
    "student": ["Amy", "Cy", "Dee"],
    "score": [80, 85, 92],
})
print(pd.merge(students1, students2, on="student", how="inner"))
print(pd.merge(students1, students2, on="student", how="outer"))

合并后的行数取决于键是否重复：两侧键唯一时通常一键对应一行；一侧键重复会形成一对多或多对一；两侧都重复时，同一键的匹配组合会产生多行。若两表有同名非键列，可用 suffixes 区分来源。

In [ ]:
a = pd.DataFrame({"id": [1, 2], "score": [80, 90]})
b = pd.DataFrame({"id": [1, 2], "score": [82, 88]})
print(pd.merge(a, b, on="id", suffixes=("_left", "_right")))

## 3. GroupBy：分组计算

GroupBy 按一个或多个键将数据拆成组，再对每组聚合。聚合将每组压缩成汇总值；source 展示 `mean`、`min`、`max`、`median` 和 `describe`。可对不同列指定不同聚合函数。

In [ ]:
scores = pd.DataFrame({
    "school": ["A", "A", "B", "B"],
    "class": ["1", "2", "1", "2"],
    "age": [12, 13, 14, 12],
    "bmi": [18.2, 19.1, 21.0, 17.8],
})
print(scores.groupby("school")["age"].agg(["mean", "min", "max"]))
print(scores.groupby(["school", "class"]).agg({"age": ["mean", "max"], "bmi": "mean"}))

GroupBy 还支持不同用途：
- `aggregate/agg`：每组产生汇总值；
- `filter`：按组条件筛选组；
- `transform`：把组级计算结果映回原行数；
- `apply`：对每组应用指定操作。
分组键可以是列，也可以按索引层级或映射规则指定。

In [ ]:
group_mean = scores.groupby("school")["bmi"].transform("mean")
centered = scores["bmi"] - group_mean
print(centered)

## 4. Pivot table：分组汇总的表格视图

透视表以行索引和列标签划分组别，对指定数值列聚合。常用参数包括 `values`（汇总列）、`index`（行分组）、`columns`（列分组）、`aggfunc`（聚合函数）、`fill_value`（填充空格）和 `margins`（增加总计）。这与 GroupBy 后再 unstack 的结果可以表达相似汇总。

In [ ]:
titanic = pd.DataFrame({
    "sex": ["female", "male", "female", "male"],
    "class": ["1st", "1st", "2nd", "2nd"],
    "survived": [1, 0, 1, 0],
    "fare": [80, 50, 30, 20],
})
table = titanic.pivot_table(
    values="survived",
    index="sex",
    columns="class",
    aggfunc="mean",
    margins=True,
)
print(table)

### 补充：数组拼接与分组/透视题目

原讲义也先展示 NumPy 数组的 `concatenate`，随后讲 Pandas Series/DataFrame 的 `concat`；两者都是沿轴组合数据。GroupBy 的练习以 school code、class 分组统计年龄的 mean/min/max，并筛出年龄条件后计算每校 BMI 平均值。透视表练习按 gender、年龄区间（0–18、18–40、40–65、65–80）与 class 汇总 Titanic 存活率。

In [ ]:
import numpy as np
array_a = np.array([1, 2])
array_b = np.array([3, 4])
print(np.concatenate([array_a, array_b]))

source 还展示 DataFrame 的 `filter(like=...)` 按列名筛选，例如只选名称中含有指定文字的列；它与 GroupBy 的 `filter`（按组条件保留整组）用途不同。

拼接练习还包括把 Series 加入 DataFrame，并重设行索引；表连接练习则从按列拼接进一步转为按学生姓名匹配，再处理外连接产生的缺失项。

## 本周回顾

concat 沿轴连接数据；merge 按键匹配并由连接类型决定保留哪些键；GroupBy 拆分数据后聚合、筛选、变换或应用函数；pivot_table 将分组结果排列成便于阅读的行列交叉表。